# P2D

In [6]:
# LOAD time_series

import pandas as pd
from P2D import dataloader
from P2D.utils import spacecraft_ID
import numpy as np
from scipy.stats import linregress


time_series_0 = pd.read_parquet('combined_model_superlong_2007_2008_1d_smooth.parquet')
time_series_1 = pd.read_parquet('combined_model_superlong_2008_2015_1d_smooth.parquet')
time_series_2 = pd.read_parquet('single_model_superlong.parquet')
time_series_3 = pd.read_parquet('combined_model_superlong_2023_2026_1d_smooth.parquet')

time_series_1 = time_series_1.loc[time_series_0.index[-1]:]
time_series_2 = time_series_2.loc[time_series_1.index[-1]:]
time_series_3 = time_series_3.loc[time_series_2.index[-1]:]
time_series = pd.concat([time_series_0, time_series_1, time_series_2, time_series_3]).drop_duplicates(keep='first')




timeframe = ['2012-6-1', '2017-5-1']
#timeframe = None

if timeframe is not None:
    time_series = time_series[timeframe[0]:timeframe[1]]

start_time = time_series.index.min().strftime('%Y-%m-%d-%H:%M')
end_time = time_series.index.max().strftime('%Y-%m-%d-%H:%M')

# Load spacecraft data
spacecraft_data = dataloader.load(spacecraft='OMNI', timerange=[start_time, end_time], cadence='1h')
spacecraft_data = spacecraft_data[~spacecraft_data.index.duplicated(keep='first')]


remove_cmes=False

if remove_cmes:
    cmes = pd.read_excel('HELIO4CAST_ICMECAT_v23 (1).xlsx')

    spacecraft = ['Wind', 'STEREO-A', 'SolarOrbiter', 'PSP']
    my_spacecraft = ['OMNI', 'STEREO-A', 'SolO', 'PSP']

    for i, sc in enumerate(spacecraft):
        print(sc, ' => ', my_spacecraft[i])
        start_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['icme_start_time']).reset_index()['icme_start_time'] - pd.Timedelta(value='16h')
        end_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['mo_end_time']).reset_index()['mo_end_time'] + pd.Timedelta(value='16h')

        mask = time_series['Observer_SC']==spacecraft_ID(my_spacecraft[i], ID_number=True)

        for j in range(len(start_times)):
            
            start = start_times[j].round("h").tz_localize(None)
            end = end_times[j].round("h").tz_localize(None)

            time_mask = (mask.index < end) & (mask.index > start)


            print(start, end, time_mask.sum())
            time_series[time_mask] = np.nan
            
if remove_cmes:
    cmes = pd.read_excel('HELIO4CAST_ICMECAT_v23 (1).xlsx')

    spacecraft = ['Wind', 'STEREO-A', 'SolarOrbiter', 'PSP']
    my_spacecraft = ['OMNI', 'STEREO-A', 'SolO', 'PSP']

    for i, sc in enumerate(spacecraft):
        print(sc, ' => ', my_spacecraft[i])
        start_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['icme_start_time']).reset_index()['icme_start_time'] - pd.Timedelta(value='16h')
        end_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['mo_end_time']).reset_index()['mo_end_time'] + pd.Timedelta(value='16h')

        mask = spacecraft_data['Spacecraft_ID']==spacecraft_ID(my_spacecraft[i], ID_number=True)

        for j in range(len(start_times)):
            
            start = start_times[j].round("h").tz_localize(None)
            end = end_times[j].round("h").tz_localize(None)

            time_mask = (mask.index < end) & (mask.index > start)


            print(start, end, time_mask.sum())
            spacecraft_data[time_mask] = np.nan



#Calculate METRICS

## V

# POINT TO POINT

time_series['ΔV'] = time_series['V'] - time_series.index.map(spacecraft_data['V'])
ME_V = time_series['ΔV'].mean()

time_series['ΔV'] = (time_series['V'] - time_series.index.map(spacecraft_data['V']))**2
MSE_V = time_series['ΔV'].mean()

time_series['V_MAE'] = abs(time_series['V'] - time_series.index.map(spacecraft_data['V']))
MAE_V = time_series['V_MAE'].mean()

RMSE_V = np.sqrt(np.mean((time_series['V'] - time_series.index.map(spacecraft_data['V']))**2))
#RMSE_V = time_series['V_RMSE'].mean()

# time_series['HITS_V'] = abs(time_series['V'] - time_series.index.map(spacecraft_data['V'])) < time_series['V_ERROR']
# HITS_V = time_series['HITS_V'].sum()/len(time_series)*100


x = time_series['V'].sort_index()
y = x.index.map(spacecraft_data['V'])
mask = ~np.isnan(x) & ~np.isnan(y)
slope, intercept, r_value_V, p_value, std_err = linregress(x[mask], y[mask])


# BINARY

threshold_V = time_series['V'].mean()
#threshold_V =450

time_series['TP'] = (time_series['V'] >threshold_V) & (time_series.index.map(spacecraft_data['V'])>threshold_V)
time_series['FP'] = (time_series['V'] >threshold_V) & (time_series.index.map(spacecraft_data['V'])<threshold_V)
time_series['TN'] = (time_series['V'] <threshold_V) & (time_series.index.map(spacecraft_data['V'])<threshold_V)
time_series['FN'] = (time_series['V'] <threshold_V) & (time_series.index.map(spacecraft_data['V'])>threshold_V)

TPR = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FN'].sum())
FPR = time_series['FP'].sum() /(time_series['FP'].sum()+time_series['TN'].sum())
TS = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FP'].sum()+time_series['FN'].sum())
TSS = TPR-FPR
Bias = (time_series['TP'].sum()+time_series['FP'].sum()) /(time_series['TP'].sum()+time_series['FN'].sum())

## N

time_series['ΔN'] = time_series['N'] - time_series.index.map(spacecraft_data['N'])
ME_N = time_series['ΔN'].mean()

time_series['ΔN'] = (time_series['N'] - time_series.index.map(spacecraft_data['N']))**2
MSE_N = time_series['ΔN'].mean()

time_series['N_MAE'] = abs(time_series['N'] - time_series.index.map(spacecraft_data['N']))
MAE_N = time_series['N_MAE'].mean()

RMSE_N = np.sqrt(np.mean((time_series['N'] - time_series.index.map(spacecraft_data['N']))**2))
#RMSE_N = time_series['N_RMSE'].mean()

time_series['HITS_N'] = abs(time_series['N'] - time_series.index.map(spacecraft_data['N'])) < time_series['N_ERROR']
HITS_N = time_series['HITS_N'].sum()/len(time_series)*100

x = time_series['N'].sort_index()
y = x.index.map(spacecraft_data['N'])
mask = ~np.isnan(x) & ~np.isnan(y)
slope, intercept, r_value_N, p_value, std_err = linregress(x[mask], y[mask])

# BINARY

threshold_N = time_series['N'].mean()

time_series['TP'] = (time_series['N'] >threshold_N) & (time_series.index.map(spacecraft_data['N'])>threshold_N)
time_series['FP'] = (time_series['N'] >threshold_N) & (time_series.index.map(spacecraft_data['N'])<threshold_N)
time_series['TN'] = (time_series['N'] <threshold_N) & (time_series.index.map(spacecraft_data['N'])<threshold_N)
time_series['FN'] = (time_series['N'] <threshold_N) & (time_series.index.map(spacecraft_data['N'])>threshold_N)

TPR_N = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FN'].sum())
FPR_N  = time_series['FP'].sum() /(time_series['FP'].sum()+time_series['TN'].sum())
TS_N  = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FP'].sum()+time_series['FN'].sum())
TSS_N  = TPR_N -FPR_N 
Bias_N  = (time_series['TP'].sum()+time_series['FP'].sum()) /(time_series['TP'].sum()+time_series['FN'].sum())



errors = {'MAE_V': MAE_V.round(2),
          'RMSE_V': RMSE_V.round(2),
          'ME_V': ME_V.round(2),
          'MSE_V': MSE_V.round(2),
          #'Accuracy_V': HITS_V.round(2),
          'cc_V': r_value_V.round(2),
          'TPR' : TPR.round(2),
          'FPR' :FPR.round(2),
          'TS' :TS.round(2),
          'TSS' :TSS.round(2),
          'Bias' :Bias.round(2),

          
          'MAE_N': MAE_N.round(2),
          'RMSE_N': RMSE_N.round(2),
          'ME_N': ME_N.round(2),
          'MSE_N': MSE_N.round(2),
          #'Accuracy_N': HITS_N.round(2),
          'cc_N': r_value_N.round(2),
          'TPR_N' : TPR_N.round(2),
          'FPR_N' :FPR_N.round(2),
          'TS_N' :TS_N.round(2),
          'TSS_N' :TSS_N.round(2),
          'Bias_N' :Bias_N.round(2),}

errors


{'MAE_V': 67.79,
 'RMSE_V': 91.44,
 'ME_V': -1.0,
 'MSE_V': 8361.16,
 'cc_V': 0.5,
 'TPR': 0.62,
 'FPR': 0.24,
 'TS': 0.46,
 'TSS': 0.37,
 'Bias': 0.95,
 'MAE_N': 3.51,
 'RMSE_N': 5.56,
 'ME_N': -0.67,
 'MSE_N': 30.89,
 'cc_N': 0.21,
 'TPR_N': 0.46,
 'FPR_N': 0.26,
 'TS_N': 0.34,
 'TSS_N': 0.2,
 'Bias_N': 0.82}

# P27

In [ ]:


# LOAD time_series


import pandas as pd
from P2D import dataloader
from P2D.utils import spacecraft_ID
import numpy as np
from scipy.stats import linregress


time_series = pd.read_parquet('27_27d_persistence_1h_smooth_whole_range.parquet')


timeframe = ['2012-6-1', '2017-5-1']
#timeframe = None

if timeframe is not None:
    time_series = time_series[timeframe[0]:timeframe[1]]

start_time = time_series.index.min().strftime('%Y-%m-%d-%H:%M')
end_time = time_series.index.max().strftime('%Y-%m-%d-%H:%M')

# Load spacecraft data
spacecraft_data = dataloader.load(spacecraft='OMNI', timerange=[start_time, end_time], cadence='1h')
spacecraft_data = spacecraft_data[~spacecraft_data.index.duplicated(keep='first')]



remove_cmes=False
if remove_cmes:
    cmes = pd.read_excel('HELIO4CAST_ICMECAT_v23 (1).xlsx')

    spacecraft = ['Wind', 'STEREO-A', 'SolarOrbiter', 'PSP']
    my_spacecraft = ['OMNI', 'STEREO-A', 'SolO', 'PSP']

    for i, sc in enumerate(spacecraft):
        print(sc, ' => ', my_spacecraft[i])
        start_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['icme_start_time']).reset_index()['icme_start_time'] - pd.Timedelta(value='16h')
        end_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['mo_end_time']).reset_index()['mo_end_time'] + pd.Timedelta(value='16h')

        mask = time_series['Observer_SC']==spacecraft_ID(my_spacecraft[i], ID_number=True)

        for j in range(len(start_times)):
            
            start = start_times[j].round("h").tz_localize(None)
            end = end_times[j].round("h").tz_localize(None)

            time_mask = (mask.index < end) & (mask.index > start)


            print(start, end, time_mask.sum())
            time_series[time_mask] = np.nan
            
if remove_cmes:
    cmes = pd.read_excel('HELIO4CAST_ICMECAT_v23 (1).xlsx')

    spacecraft = ['Wind', 'STEREO-A', 'SolarOrbiter', 'PSP']
    my_spacecraft = ['OMNI', 'STEREO-A', 'SolO', 'PSP']

    for i, sc in enumerate(spacecraft):
        print(sc, ' => ', my_spacecraft[i])
        start_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['icme_start_time']).reset_index()['icme_start_time'] - pd.Timedelta(value='16h')
        end_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['mo_end_time']).reset_index()['mo_end_time'] + pd.Timedelta(value='16h')

        mask = spacecraft_data['Spacecraft_ID']==spacecraft_ID(my_spacecraft[i], ID_number=True)

        for j in range(len(start_times)):
            
            start = start_times[j].round("h").tz_localize(None)
            end = end_times[j].round("h").tz_localize(None)

            time_mask = (mask.index < end) & (mask.index > start)


            print(start, end, time_mask.sum())
            spacecraft_data[time_mask] = np.nan


#Calculate METRICS

## V

# POINT TO POINT

time_series['ΔV'] = time_series['V'] - time_series.index.map(spacecraft_data['V'])
ME_V = time_series['ΔV'].mean()

time_series['ΔV'] = (time_series['V'] - time_series.index.map(spacecraft_data['V']))**2
MSE_V = time_series['ΔV'].mean()

time_series['V_MAE'] = abs(time_series['V'] - time_series.index.map(spacecraft_data['V']))
MAE_V = time_series['V_MAE'].mean()

RMSE_V = np.sqrt(np.mean((time_series['V'] - time_series.index.map(spacecraft_data['V']))**2))
#RMSE_V = time_series['V_RMSE'].mean()

# time_series['HITS_V'] = abs(time_series['V'] - time_series.index.map(spacecraft_data['V'])) < time_series['V_ERROR']
# HITS_V = time_series['HITS_V'].sum()/len(time_series)*100


x = time_series['V'].sort_index()
y = x.index.map(spacecraft_data['V'])
mask = ~np.isnan(x) & ~np.isnan(y)
slope, intercept, r_value_V, p_value, std_err = linregress(x[mask], y[mask])


# BINARY

#threshold_V = time_series['V'].mean()
threshold_V = 450

time_series['TP'] = (time_series['V'] >threshold_V) & (time_series.index.map(spacecraft_data['V'])>threshold_V)
time_series['FP'] = (time_series['V'] >threshold_V) & (time_series.index.map(spacecraft_data['V'])<threshold_V)
time_series['TN'] = (time_series['V'] <threshold_V) & (time_series.index.map(spacecraft_data['V'])<threshold_V)
time_series['FN'] = (time_series['V'] <threshold_V) & (time_series.index.map(spacecraft_data['V'])>threshold_V)

TPR = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FN'].sum())
FPR = time_series['FP'].sum() /(time_series['FP'].sum()+time_series['TN'].sum())
TS = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FP'].sum()+time_series['FN'].sum())
TSS = TPR-FPR
Bias = (time_series['TP'].sum()+time_series['FP'].sum()) /(time_series['TP'].sum()+time_series['FN'].sum())

## N

time_series['ΔN'] = time_series['N'] - time_series.index.map(spacecraft_data['N'])
ME_N = time_series['ΔN'].mean()

time_series['ΔN'] = (time_series['N'] - time_series.index.map(spacecraft_data['N']))**2
MSE_N = time_series['ΔN'].mean()

time_series['N_MAE'] = abs(time_series['N'] - time_series.index.map(spacecraft_data['N']))
MAE_N = time_series['N_MAE'].mean()

RMSE_N = np.sqrt(np.mean((time_series['N'] - time_series.index.map(spacecraft_data['N']))**2))
#RMSE_N = time_series['N_RMSE'].mean()

time_series['HITS_N'] = abs(time_series['N'] - time_series.index.map(spacecraft_data['N'])) < time_series['N_ERROR']
HITS_N = time_series['HITS_N'].sum()/len(time_series)*100

x = time_series['N'].sort_index()
y = x.index.map(spacecraft_data['N'])
mask = ~np.isnan(x) & ~np.isnan(y)
slope, intercept, r_value_N, p_value, std_err = linregress(x[mask], y[mask])

# BINARY

threshold_N = time_series['N'].mean()

time_series['TP'] = (time_series['N'] >threshold_N) & (time_series.index.map(spacecraft_data['N'])>threshold_N)
time_series['FP'] = (time_series['N'] >threshold_N) & (time_series.index.map(spacecraft_data['N'])<threshold_N)
time_series['TN'] = (time_series['N'] <threshold_N) & (time_series.index.map(spacecraft_data['N'])<threshold_N)
time_series['FN'] = (time_series['N'] <threshold_N) & (time_series.index.map(spacecraft_data['N'])>threshold_N)

TPR_N = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FN'].sum())
FPR_N  = time_series['FP'].sum() /(time_series['FP'].sum()+time_series['TN'].sum())
TS_N  = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FP'].sum()+time_series['FN'].sum())
TSS_N  = TPR_N -FPR_N 
Bias_N  = (time_series['TP'].sum()+time_series['FP'].sum()) /(time_series['TP'].sum()+time_series['FN'].sum())



errors = {'MAE_V': MAE_V.round(2),
          'RMSE_V': RMSE_V.round(2),
          'ME_V': ME_V.round(2),
          'MSE_V': MSE_V.round(2),
          #'Accuracy_V': HITS_V.round(2),
          'cc_V': r_value_V.round(2),
          'TPR' : TPR.round(2),
          'FPR' :FPR.round(2),
          'TS' :TS.round(2),
          'TSS' :TSS.round(2),
          'Bias' :Bias.round(2),

          
          'MAE_N': MAE_N.round(2),
          'RMSE_N': RMSE_N.round(2),
          'ME_N': ME_N.round(2),
          'MSE_N': MSE_N.round(2),
          #'Accuracy_N': HITS_N.round(2),
          'cc_N': r_value_N.round(2),
          'TPR_N' : TPR_N.round(2),
          'FPR_N' :FPR_N.round(2),
          'TS_N' :TS_N.round(2),
          'TSS_N' :TSS_N.round(2),
          'Bias_N' :Bias_N.round(2),}

errors


{'MAE_V': 73.47,
 'RMSE_V': 99.55,
 'ME_V': -0.6,
 'MSE_V': 9910.39,
 'cc_V': 0.44,
 'TPR': 0.55,
 'FPR': 0.2,
 'TS': 0.38,
 'TSS': 0.35,
 'Bias': 1.0,
 'MAE_N': 3.81,
 'RMSE_N': 5.93,
 'ME_N': -0.05,
 'MSE_N': 35.14,
 'cc_N': 0.2,
 'TPR_N': 0.48,
 'FPR_N': 0.28,
 'TS_N': 0.32,
 'TSS_N': 0.2,
 'Bias_N': 0.98}

# HUXt

In [5]:
# LOAD time_series

import pandas as pd
from P2D import dataloader
from P2D.utils import spacecraft_ID
import numpy as np
from scipy.stats import linregress


# HUXt

time_series = pd.read_csv('insitu_huxt_forecasts_4day_leadtime.csv')
time_series['time'] = pd.to_datetime(time_series['time'])
time_series.set_index('time', inplace=True)
time_series['V'] = time_series['vsw']
time_series['Observer_SC'] = 'OMNI'


timeframe = ['2012-6-1', '2017-5-1']
#timeframe = None

if timeframe is not None:
    time_series = time_series[timeframe[0]:timeframe[1]]

start_time = time_series.index.min().strftime('%Y-%m-%d-%H:%M')
end_time = time_series.index.max().strftime('%Y-%m-%d-%H:%M')

# Load spacecraft data
spacecraft_data = dataloader.load(spacecraft='OMNI', timerange=[start_time, end_time], cadence='1h')
spacecraft_data = spacecraft_data[~spacecraft_data.index.duplicated(keep='first')]



remove_cmes=False
if remove_cmes:
    cmes = pd.read_excel('HELIO4CAST_ICMECAT_v23 (1).xlsx')

    spacecraft = ['Wind', 'STEREO-A', 'SolarOrbiter', 'PSP']
    my_spacecraft = ['OMNI', 'STEREO-A', 'SolO', 'PSP']

    for i, sc in enumerate(spacecraft):
        print(sc, ' => ', my_spacecraft[i])
        start_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['icme_start_time']).reset_index()['icme_start_time'] - pd.Timedelta(value='16h')
        end_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['mo_end_time']).reset_index()['mo_end_time'] + pd.Timedelta(value='16h')

        mask = time_series['Observer_SC']==spacecraft_ID(my_spacecraft[i], ID_number=True)

        for j in range(len(start_times)):
            
            start = start_times[j].round("h").tz_localize(None)
            end = end_times[j].round("h").tz_localize(None)

            time_mask = (mask.index < end) & (mask.index > start)


            print(start, end, time_mask.sum())
            time_series[time_mask] = np.nan
            
if remove_cmes:
    cmes = pd.read_excel('HELIO4CAST_ICMECAT_v23 (1).xlsx')

    spacecraft = ['Wind', 'STEREO-A', 'SolarOrbiter', 'PSP']
    my_spacecraft = ['OMNI', 'STEREO-A', 'SolO', 'PSP']

    for i, sc in enumerate(spacecraft):
        print(sc, ' => ', my_spacecraft[i])
        start_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['icme_start_time']).reset_index()['icme_start_time'] - pd.Timedelta(value='16h')
        end_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['mo_end_time']).reset_index()['mo_end_time'] + pd.Timedelta(value='16h')

        mask = spacecraft_data['Spacecraft_ID']==spacecraft_ID(my_spacecraft[i], ID_number=True)

        for j in range(len(start_times)):
            
            start = start_times[j].round("h").tz_localize(None)
            end = end_times[j].round("h").tz_localize(None)

            time_mask = (mask.index < end) & (mask.index > start)


            print(start, end, time_mask.sum())
            spacecraft_data[time_mask] = np.nan


## V

# POINT TO POINT

time_series_with_new_index = time_series[['V']].resample('1h').mean()
time_series_with_new_index

time_series_with_new_index['ΔV'] = time_series_with_new_index['V'] - time_series_with_new_index.index.map(spacecraft_data['V'])
ME_V = time_series_with_new_index['ΔV'].mean()

time_series_with_new_index['ΔV'] = (time_series_with_new_index['V'] - time_series_with_new_index.index.map(spacecraft_data['V']))**2
MSE_V = time_series_with_new_index['ΔV'].mean()

time_series_with_new_index['V_MAE'] = abs(time_series_with_new_index['V'] - time_series_with_new_index.index.map(spacecraft_data['V']))
MAE_V = time_series_with_new_index['V_MAE'].mean()

RMSE_V = np.sqrt(np.mean((time_series_with_new_index['V'] - time_series_with_new_index.index.map(spacecraft_data['V']))**2))

x = time_series_with_new_index['V'].sort_index()
y = x.index.map(spacecraft_data['V'])
mask = ~np.isnan(x) & ~np.isnan(y)
slope, intercept, r_value_V, p_value, std_err = linregress(x[mask], y[mask])


# BINARY

#threshold_V = time_series_with_new_index['V'].mean()
threshold_V = 450

time_series_with_new_index['TP'] = (time_series_with_new_index['V'] >threshold_V) & (time_series_with_new_index.index.map(spacecraft_data['V'])>threshold_V)
time_series_with_new_index['FP'] = (time_series_with_new_index['V'] >threshold_V) & (time_series_with_new_index.index.map(spacecraft_data['V'])<threshold_V)
time_series_with_new_index['TN'] = (time_series_with_new_index['V'] <threshold_V) & (time_series_with_new_index.index.map(spacecraft_data['V'])<threshold_V)
time_series_with_new_index['FN'] = (time_series_with_new_index['V'] <threshold_V) & (time_series_with_new_index.index.map(spacecraft_data['V'])>threshold_V)

TPR = time_series_with_new_index['TP'].sum() /(time_series_with_new_index['TP'].sum()+time_series_with_new_index['FN'].sum())
FPR = time_series_with_new_index['FP'].sum() /(time_series_with_new_index['FP'].sum()+time_series_with_new_index['TN'].sum())
TS = time_series_with_new_index['TP'].sum() /(time_series_with_new_index['TP'].sum()+time_series_with_new_index['FP'].sum()+time_series_with_new_index['FN'].sum())
TSS = TPR-FPR
Bias = (time_series_with_new_index['TP'].sum()+time_series_with_new_index['FP'].sum()) /(time_series_with_new_index['TP'].sum()+time_series_with_new_index['FN'].sum())



errors = {'MAE_V': MAE_V.round(2),
          'RMSE_V': RMSE_V.round(2),
          'ME_V': ME_V.round(2),
          'MSE_V': MSE_V.round(2),
          'cc_V': r_value_V.round(2),
          'TPR' : TPR.round(2),
          'FPR' :FPR.round(2),
          'TS' :TS.round(2),
          'TSS' :TSS.round(2),
          'Bias' :Bias.round(2)}#,


errors


{'MAE_V': 71.06,
 'RMSE_V': 96.36,
 'ME_V': -3.41,
 'MSE_V': 9285.55,
 'cc_V': 0.48,
 'TPR': 0.56,
 'FPR': 0.18,
 'TS': 0.4,
 'TSS': 0.38,
 'Bias': 0.96}

# ESWF

In [4]:
import pandas as pd
from P2D import dataloader
from P2D.utils import spacecraft_ID
import numpy as np
from scipy.stats import linregress


time_series= pd.read_csv('forecast_archive1.txt')
time_series['DATE'] = pd.to_datetime(time_series['DATE'])
time_series['V'] = time_series[' V']
time_series.set_index('DATE', inplace=True)
for i in range(2,10):
    #print(i)
    time_series_addon = pd.read_csv('forecast_archive'+str(i)+'.txt')
    time_series_addon['DATE'] = pd.to_datetime(time_series_addon['DATE'])
    time_series_addon.set_index('DATE', inplace=True)
    time_series_addon['V'] = time_series_addon[' V']
    time_series_addon = time_series_addon.resample('3H').mean()
    time_series_addon = time_series_addon.loc[time_series.index[-1]:]
    
    time_series = pd.concat([time_series, time_series_addon])


timeframe = ['2012-6-1', '2017-5-1']
#timeframe = None

if timeframe is not None:
    time_series = time_series[timeframe[0]:timeframe[1]]

start_time = time_series.index.min().strftime('%Y-%m-%d-%H:%M')
end_time = time_series.index.max().strftime('%Y-%m-%d-%H:%M')

# Load spacecraft data
spacecraft_data = dataloader.load(spacecraft='OMNI', timerange=[start_time, end_time], cadence='1h')
spacecraft_data = spacecraft_data[~spacecraft_data.index.duplicated(keep='first')]



remove_cmes=False
        
if remove_cmes:
    cmes = pd.read_excel('HELIO4CAST_ICMECAT_v23 (1).xlsx')

    spacecraft = ['Wind', 'STEREO-A', 'SolarOrbiter', 'PSP']
    my_spacecraft = ['OMNI', 'STEREO-A', 'SolO', 'PSP']

    for i, sc in enumerate(spacecraft):
        print(sc, ' => ', my_spacecraft[i])
        start_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['icme_start_time']).reset_index()['icme_start_time'] - pd.Timedelta(value='16h')
        end_times = pd.to_datetime(cmes.loc[cmes['sc_insitu']==sc]['mo_end_time']).reset_index()['mo_end_time'] + pd.Timedelta(value='16h')

        mask = spacecraft_data['Spacecraft_ID']==spacecraft_ID(my_spacecraft[i], ID_number=True)

        for j in range(len(start_times)):
            
            start = start_times[j].round("h").tz_localize(None)
            end = end_times[j].round("h").tz_localize(None)

            time_mask = (mask.index < end) & (mask.index > start)


            print(start, end, time_mask.sum())
            spacecraft_data[time_mask] = np.nan




## V

# POINT TO POINT

time_series['ΔV'] = time_series['V'] - time_series.index.map(spacecraft_data['V'])
ME_V = time_series['ΔV'].mean()

time_series['ΔV'] = (time_series['V'] - time_series.index.map(spacecraft_data['V']))**2
MSE_V = time_series['ΔV'].mean()

time_series['V_MAE'] = abs(time_series['V'] - time_series.index.map(spacecraft_data['V']))
MAE_V = time_series['V_MAE'].mean()

RMSE_V = np.sqrt(np.mean((time_series['V'] - time_series.index.map(spacecraft_data['V']))**2))


x = time_series['V'].sort_index()
y = x.index.map(spacecraft_data['V'])
mask = ~np.isnan(x) & ~np.isnan(y)
slope, intercept, r_value_V, p_value, std_err = linregress(x[mask], y[mask])


# BINARY

threshold_V = 450

time_series['TP'] = (time_series['V'] >threshold_V) & (time_series.index.map(spacecraft_data['V'])>threshold_V)
time_series['FP'] = (time_series['V'] >threshold_V) & (time_series.index.map(spacecraft_data['V'])<threshold_V)
time_series['TN'] = (time_series['V'] <threshold_V) & (time_series.index.map(spacecraft_data['V'])<threshold_V)
time_series['FN'] = (time_series['V'] <threshold_V) & (time_series.index.map(spacecraft_data['V'])>threshold_V)

TPR = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FN'].sum())
FPR = time_series['FP'].sum() /(time_series['FP'].sum()+time_series['TN'].sum())
TS = time_series['TP'].sum() /(time_series['TP'].sum()+time_series['FP'].sum()+time_series['FN'].sum())
TSS = TPR-FPR
Bias = (time_series['TP'].sum()+time_series['FP'].sum()) /(time_series['TP'].sum()+time_series['FN'].sum())



errors = {'MAE_V': MAE_V.round(2),
          'RMSE_V': RMSE_V.round(2),
          'ME_V': ME_V.round(2),
          'MSE_V': MSE_V.round(2),
          'cc_V': r_value_V.round(2),
          'TPR' : TPR.round(2),
          'FPR' :FPR.round(2),
          'TS' :TS.round(2),
          'TSS' :TSS.round(2),
          'Bias' :Bias.round(2),
}

errors


{'MAE_V': 82.42,
 'RMSE_V': 108.46,
 'ME_V': -24.85,
 'MSE_V': 11764.23,
 'cc_V': 0.43,
 'TPR': 0.47,
 'FPR': 0.18,
 'TS': 0.37,
 'TSS': 0.28,
 'Bias': 0.74}